# From simulated training to historical S&P 500 replay

Notebook 02 established which hedging strategies work when the market really is a rough-Bergomi model. This notebook asks whether that advantage survives in real markets. The design separates two roles strictly:

1. **Training uses simulations only.** The learned hedgers are trained on rough-Bergomi paths whose parameters are calibrated to S&P 500 and VIX history: Hurst exponent $H = 0.107$, vol-of-vol $\eta = 1.70$, spot–vol correlation $\rho = -0.43$ (estimated from realised variance), and forward variance level $\xi_0 = 0.0386$, the average of $(\text{VIX}/100)^2$ over 4,025 days.
2. **Evaluation uses real prices only.** The trained policies are then replayed, unchanged, on historical S&P 500 windows that they never saw, next to classical hedges that use the VIX as their implied volatility.

**The replayed position.** At the start of each window the hedger sells an at-the-money call on the index, priced with Black–Scholes at the VIX level of that day, and then hedges at each rebalancing date with 10 bp of proportional cost per side. The loss of a window is

$$L = \text{payoff} - \text{premium} - \text{hedging gains} + \text{trading costs},$$

so a negative loss is a profit. Every arm sees exactly the same windows, premium and payoff, which makes the comparisons paired.

**A note on the VIX.** The VIX is a 30-day implied volatility, smoothed and forward-looking. It is used here as the implied-volatility input of the classical hedges and for pricing the premium, not as instantaneous variance. The learned policies receive only the price path, as in training.


In [1]:
from pathlib import Path
import json

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
run_dir = ROOT / 'results' / 'nb03' / 'runs' / 'l4-20261003-full-04'
hedging = json.loads((run_dir / 'hedging.json').read_text())
intraday = json.loads((run_dir / 'intraday_rehedging.json').read_text())
train_log = json.loads((run_dir / 'train_log.json').read_text())
print('Device:', train_log['gpu_name'])
print('Training units:', len(train_log['units']))
print('Historical windows:', hedging['summary'][0]['n_windows'])

Device: NVIDIA L4
Training units: 35
Historical windows: 191


## 1. Simulated training design

The NVIDIA L4 run trained **35 units**:

| Block | Units | Time grid | Arms |
|---|---|---|---|
| Daily | 15 (3 architectures × 5 seeds) | 21 trading days, daily rebalancing | FFN, GRU, sig2 |
| Intraday | 20 (4 frequencies × 5 seeds) | 5 trading days, 1, 2, 4 or 7 rebalances per day | GRU |

Each unit used 65,536 training and 16,384 validation paths, batches of 4,096, Adam with learning rate $5 \times 10^{-3}$, up to 50 epochs and early stopping after 10 epochs without improvement, all at 10 bp cost. The whole run took 230 s of wall time on the L4 with 4 parallel workers. A single unit took a median 24 s and at most 130 MiB of GPU memory.

Before any replay, a **leakage probe** perturbs future prices and checks that no policy's earlier positions change. All daily and intraday policies passed with a maximum difference of exactly 0, while perturbing past prices did change positions, which shows the probe has power. The historical replay below is therefore genuinely out of sample: no real price ever entered training.


## 2. Context from the simulation study: the leaderboard

![Figure 5.1: simulated CVaR95 leaderboard](../figures/fig-5.1.png)

**What it shows.** The notebook-02 leaderboard, repeated here as the point of comparison. On simulated test paths every learned cell beats Black–Scholes delta (0.0620), and the best recipes reach about 0.0576.

**Why it is here.** It sets the expectation that the historical replay tests. If the simulated model were a good description of the market, the learned hedgers should keep a tail-risk advantage of several percent on real windows. *Data:* `../figures/fig-5.1.csv`.


## 3. Context from the simulation study: paired gaps to delta

![Figure 5.2: simulated paired CVaR gaps](../figures/fig-5.2.png)

**What it shows.** Paired CVaR95 differences between learned hedges and delta on 20,000 shared simulated paths, with Holm-adjusted $p$-values. All 15 large-profile cells are significantly better than delta, by 0.0015 to 0.0044.

**Why it is here.** These are the effect sizes, with near-certain statistical support, that the real-data replay is about to put under pressure. Note the scale: the whole advantage is a few tenths of a percent of notional. *Data:* `../figures/fig-5.2.csv`.


## 4. Context from the simulation study: learning dynamics

![Figure 5.3: simulated validation curves](../figures/fig-5.3.png)

**What it shows.** Validation CVaR95 during training for each recipe. Most of the improvement arrives in the first 10% of steps, and the long recipes keep polishing the last fractions of a basis point.

**Why it is here.** The policies replayed below were trained with a short large-batch recipe (50 epochs, batches of 4,096), similar to R-GPU. On simulated data that recipe lands within about 0.001 of the best. Any large gap on real data therefore cannot be blamed on under-training. *Data:* `../figures/fig-5.3.csv`.


## 5. Context from the simulation study: the extreme tail

![Figure 5.15: simulated loss exceedance](../figures/fig-5.15.png)

**What it shows.** Exceedance curves on the simulated test paths for the best learned unit against the classical hedges. The learned hedge is better from the 90% to the 99.9% level, but its single worst path is worse than delta's (0.194 against 0.127).

**Why it is here.** Real markets are a stress test of exactly that extreme tail: jumps, gaps and volatility spikes that a simulator calibrated on average behaviour produces too rarely. The simulated evidence already warns that CVaR training does not bound the worst case. *Data:* `../figures/fig-5.15.csv`.


## 6. Historical replay design

**Daily windows.**
* The S&P 500 daily closes from 2010 to 2025 are cut into **191 non-overlapping windows of 21 trading days**.
* Each window starts a fresh short call, and the VIX close of each day is joined by date. No window was dropped for missing data.
* Windows are labelled **calm** if the VIX at the start is below 20 (136 windows) and **stressed** otherwise (55 windows).

**Intraday windows.**
* Hourly bars (7 per day) cover a shorter recent period. They are cut into **57 non-overlapping 5-day windows** (42 calm, 15 stressed); 5 calendar blocks with incomplete bars were dropped.
* Rebalancing at 1, 2, 4 or 7 times per day is obtained by subsampling the bars within each day.

**Statistics.**
* CVaR95 is the mean of the worst 5% of window losses: about 10 windows out of 191, or 3 out of 57. These are small samples, so uncertainty matters a great deal.
* Consecutive windows share volatility regimes, so intervals use a **moving-block bootstrap** (blocks of 4 to 6 windows, 2,000 resamples), which keeps that serial dependence instead of treating windows as independent.
* Differences to delta are paired by window and Holm-adjusted across arms.
* Learned arms are reported as the mean over five seeds, with the seed range.


## 7. Daily replay on real S&P 500 windows

![Figure 4.5: daily S&P 500 and VIX replay](../figures/fig-4.5-l4-20261003-full-04-analysis.png)

**What it shows.**
* **(a)** Histograms of window losses for all six hedges.
* **(b)** CVaR95 with block-bootstrap intervals, separately for calm and stressed windows.
* **(c)** The paired difference of each hedge's CVaR95 to delta, with Holm-adjusted $p$-values, for all windows and for each regime.

**How to read it.** In (b), lower is better. In (c), points below zero beat delta, and a $p_{\text{Holm}}$ near 1 means the difference is indistinguishable from noise.

**What we find.**

| Hedge | CVaR95, all 191 windows | calm (136) | stressed (55) | Difference to delta, all |
|---|---|---|---|---|
| Delta (VIX) | 0.0186 [0.0103, 0.0280] | 0.0137 | 0.0300 | — |
| Leland | 0.0186 | 0.0138 | 0.0299 | +0.0000 |
| Whalley–Wilmott band | 0.0180 | 0.0163 | 0.0216 | −0.0006 |
| FFN (5 seeds) | 0.0209 | 0.0146 | 0.0348 | +0.0023 |
| GRU (5 seeds) | 0.0210 | 0.0153 | 0.0339 | +0.0024 |
| sig2 (5 seeds) | 0.0204 | 0.0147 | 0.0328 | +0.0018 |

* **The simulated advantage does not transfer.** On real windows all three learned hedgers have a *higher* CVaR95 than delta, by 0.0018 to 0.0024. None of these differences is significant (Holm $p = 1.0$ for every learned arm). The point estimates are about 10–13% worse than delta, the opposite sign of the 7% gain in simulation.
* **Stress dominates the tail.** Every hedge's CVaR95 roughly doubles from calm to stressed windows, and the bootstrap intervals in stressed windows are very wide (delta: 0.0075 to 0.0393). The learned hedgers degrade more in stress (+0.0028 to +0.0048 against delta) than in calm markets (+0.0009 to +0.0016).
* **The no-trade band is the most robust in stress.** It has the lowest stressed-window CVaR95 (0.0216 against 0.0300 for delta, a gap of −0.0084). That gap is the largest of the study but still not significant ($p_{\text{Holm}} = 1.0$) with 55 windows. In calm markets it is the worst classical hedge, consistent with the simulation, where it also had the heaviest tail.
* The loss histograms (a) are nearly identical in their body. All differences live in the right tail, which is made of about 10 windows.

**Why it matters.** This is the central real-data result, and it is negative. A hedge trained to be optimal in a rough-volatility model calibrated to the same market did not beat delta on that market's history. The statistical power is limited (191 monthly windows give about 10 tail observations), so the honest conclusion is *no evidence of improvement*, with point estimates that lean against the learned hedges. *Data:* `../figures/fig-4.5-l4-20261003-full-04-analysis.csv`.


## 8. Intraday rehedging on real hourly data

![Figure 4.6: intraday rehedging replay](../figures/fig-4.6-l4-20261003-full-04-analysis.png)

**What it shows.**
* **(a)** CVaR95 against the number of rebalances per day (1, 2, 4, 7) for delta, the no-trade band and the GRU, with block-bootstrap intervals. A star marks the first frequency at which the gain from hedging more often falls below its own standard error.
* **(b)** Mean turnover (solid) and mean trading cost (dashed) against frequency.
* **(c)** The cumulative P&L paths of the three worst real windows for delta at 7 rebalances per day, with their dates.

**How to read it.** Moving right means hedging more often. Hedging error should fall with frequency while costs rise, so the curve should flatten or turn up once costs outweigh the tighter hedge.

**What we find.**

| Rebalances per day | Delta CVaR95 | Band CVaR95 | GRU CVaR95 | Delta turnover | Band turnover | GRU turnover |
|---|---|---|---|---|---|---|
| 1 | 0.0153 | 0.0130 | 0.0153 | 1.62 | 0.79 | 1.25 |
| 2 | 0.0172 | 0.0136 | 0.0123 | 1.91 | 0.97 | 1.37 |
| 4 | 0.0168 | 0.0125 | 0.0143 | 2.24 | 1.12 | 1.51 |
| 7 | 0.0111 | 0.0129 | 0.0130 | 2.52 | 1.19 | 1.58 |

* **Frequency buys little.** No arm improves steadily with more rebalancing. Delta is flat or worse from 1 to 4 rebalances per day and only improves at 7. The band is flat throughout, and the GRU is best at 2 per day (0.0123). Every interval is wide (57 windows, about 3 in the tail), so none of these differences can be told apart from noise.
* **Costs rise steadily regardless.** Delta's turnover rises 56% from 1.62 to 2.52, and its cost from 0.0016 to 0.0025 of notional. The band trades about half as much as delta at every frequency, and the GRU sits in between, the same ordering as in simulation (Figure 5.14).
* **The worst windows are market events, not hedging errors.**
  * The three worst delta windows at 7 per day are 1–7 April 2025, 8–14 April 2025 and 4–10 March 2025. They cover the early-April 2025 sell-off and its rebound.
  * In the first, all three hedges lose together after a gap down on day 2. In the second, the losses come from a final-day reversal.
  * No hedging rule in this family can trade through overnight gaps, because the hedge cannot be adjusted between closes and opens.

**Why it matters.** On real hourly data the extra trading of a higher rebalancing frequency is paid for with certainty, while its benefit is not measurable in this sample. The cost-aware arms (band and GRU) achieve similar tail risk with half to two thirds of delta's turnover. *Data:* `../figures/fig-4.6-l4-20261003-full-04-analysis.csv` and `../figures/fig-4.6-l4-20261003-full-04-analysis-failure_paths.csv`.


## 9. Interpretation: why the simulated advantage disappears

Three mechanisms, each visible in the evidence above, explain the gap between notebook 02 and this replay.

1. **Model misspecification in the tail.** The calibration matches average roughness and correlation, but real index paths contain overnight gaps, jumps and volatility spikes (April 2025) that a continuous rough-Bergomi model generates too rarely. The learned policies' small deviations from delta (Figure 5.7) are tuned to the simulated dynamics; in the real tail they become additional error instead of protection.
2. **Volatility input.** The classical hedges receive the VIX, a market-implied, forward-looking volatility. The learned policies receive only the price path and infer volatility as their training model would. In stressed windows the VIX reacts immediately, which gives the classical hedges an information advantage exactly where the tail is decided.
3. **Small tail samples.** The simulated comparisons rest on 1,000 tail paths out of 20,000. The real comparisons rest on about 10 tail windows out of 191, and about 3 out of 57 intraday. Effects of the simulated size (about 0.004) are well inside the real-data intervals, so even a genuine advantage of that size could not be detected here.

**What the real data do support.**
* The ordering of trading activity carries over from simulation to history: delta trades most, the band least, and the learned hedges in between.
* The no-trade band's robustness in stressed markets is the clearest real-data pattern, though it is not significant.
* The simulation study remains valid as a comparison of training methods. It does not show that those methods beat simple hedges on real markets.


## 10. Limits and evidence

**Limits.**
* **Sample size.** 191 monthly and 57 weekly windows give very few tail observations, and every learned-versus-delta difference has $p_{\text{Holm}} = 1.0$.
* **Proxies and contract choices.** The VIX proxies the implied volatility of a single at-the-money option. The contract is one specific choice: at the money, one month, 10 bp. Hourly data cover only a recent period, so the intraday results are dominated by 2025 events.
* **What is not measured.** The replay covers no option bid–ask spreads, financing or dividends.
* **What the conclusion is.** Neither the simulation nor the replay establishes future or investable performance. The result is the absence of evidence for an improvement over delta, not proof that one is impossible.

**Evidence kept in the repository.**
* `results/nb03/runs/l4-20261003-full-04/train_log.json`: the calibrated configuration and all 35 training units, with timings and memory.
* `hedging.json`: per-window losses of every arm and seed for the daily replay, with window dates, regimes and leakage-probe results.
* `intraday_rehedging.json`: the rehedging-frequency analysis and the failure windows.
* Each file records the SHA-256 of the market inputs it used. The raw market downloads themselves are not redistributed, and `scripts/fetch_data.py` retrieves them independently.


In [2]:
required = [run_dir / f for f in ('train_log.json', 'hedging.json', 'intraday_rehedging.json')]
required += [ROOT / 'figures' / f for f in ('fig-5.1.png', 'fig-5.2.png', 'fig-5.3.png', 'fig-5.15.png', 'fig-4.5-l4-20261003-full-04-analysis.png', 'fig-4.6-l4-20261003-full-04-analysis.png')]
missing = [p.name for p in required if not p.exists()]
assert not missing, f'Missing replay artifacts: {missing}'
print(f'Validated {len(required)} compact evidence artifacts.')

Validated 9 compact evidence artifacts.
